# Chapter 2 Companion Notebook: Lie Groups & Product Manifolds

**Textbook:** *A Lie-Interpretation of Robotics and Control*  
**Chapter 2:** Core Manifolds ($S^1/\mathrm{SO}(2)$, $S^2$, $\mathrm{SO}(3)$, $S^3$, $\mathrm{SE}(3)$ & Product Spaces)

This Colab-ready notebook implements and unit-tests:
1. Hat and vee maps $(\cdot)^\wedge : \mathbb{R}^3 \to \mathfrak{so}(3)$ and $(\cdot)^\vee : \mathfrak{so}(3) \to \mathbb{R}^3$ (`Eq. (2.1)`).
2. Matrix identities `Eq. (2.4)`–`(2.6)` on $\mathfrak{so}(3)$ and $\mathrm{SO}(3)$.
3. Rodrigues' $\exp$ and $\log$ maps on $\mathrm{SO}(3)$ (`Eq. (2.7)`–`(2.8)`), including small-angle and $\theta = \pi$ edge cases.
4. $\mathrm{SE}(3)$ $\exp$, $\log$, and Adjoint $\mathrm{Ad}_g$ (`Eq. (2.11)`–`(2.15)`).
5. Kinematics simulation and visualization of a quadrotor + suspended load on $\mathrm{SE}(3) \times S^2$ (`Eq. (2.17)`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(42)

## 1. Hat / Vee Maps and Matrix Identities Eq. (2.4)–(2.6)

In [ ]:
def hat_so3(x: np.ndarray) -> np.ndarray:
    """Hat map (·)^ : R^3 -> so(3) satisfying x^ y = x x y, Eq. (2.1)."""
    x = np.asarray(x, dtype=float).reshape(3)
    return np.array([
        [0.0, -x[2], x[1]],
        [x[2], 0.0, -x[0]],
        [-x[1], x[0], 0.0],
    ])

def vee_so3(S: np.ndarray) -> np.ndarray:
    """Vee map (·)^v : so(3) -> R^3, inverse of hat_so3, Eq. (2.1)."""
    S = np.asarray(S, dtype=float).reshape(3, 3)
    return np.array([S[2, 1], S[0, 2], S[1, 0]])

# Unit test 1: Hat/vee round-trip and cross-product equivalence
for _ in range(10):
    x = rng.normal(size=3)
    y = rng.normal(size=3)
    assert np.allclose(vee_so3(hat_so3(x)), x, atol=1e-14)
    assert np.allclose(hat_so3(x) + hat_so3(x).T, np.zeros((3, 3)), atol=1e-14)
    assert np.allclose(hat_so3(x) @ y, np.cross(x, y), atol=1e-14)
print("[PASS] Hat/vee round-trip & x^ y = x x y")

## 2. Rodrigues' Exponential and Logarithm Maps on SO(3)

In [ ]:
def exp_so3(phi: np.ndarray) -> np.ndarray:
    """Rodrigues' exponential map exp : so(3) -> SO(3), Eq. (2.7)."""
    phi = np.asarray(phi, dtype=float).reshape(3)
    theta = np.linalg.norm(phi)
    K = hat_so3(phi)
    if theta < 1e-8:
        t2 = theta * theta
        a = 1.0 - t2 / 6.0 + (t2 * t2) / 120.0
        b = 0.5 - t2 / 24.0 + (t2 * t2) / 720.0
    else:
        a = np.sin(theta) / theta
        b = (1.0 - np.cos(theta)) / (theta * theta)
    return np.eye(3) + a * K + b * (K @ K)

def log_so3(R: np.ndarray) -> np.ndarray:
    """Logarithm map log : SO(3) -> so(3) (returned as R^3 vector), Eq. (2.8)."""
    R = np.asarray(R, dtype=float).reshape(3, 3)
    cos_theta = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    if theta < 1e-8:
        t2 = theta * theta
        coeff = 0.5 + t2 / 12.0 + 7.0 * (t2 * t2) / 720.0
        return coeff * vee_so3(R - R.T)
    if np.pi - theta < 1e-6:
        # Cut locus theta ~= pi: R = -I + 2 u u^T
        M = R + np.eye(3)
        col_norms = np.linalg.norm(M, axis=0)
        idx = int(np.argmax(col_norms))
        u = M[:, idx] / col_norms[idx]
        return theta * u
    return (theta / (2.0 * np.sin(theta))) * vee_so3(R - R.T)

# Unit test 2: Matrix identities Eq. (2.4)-(2.6)
for _ in range(10):
    x = rng.normal(size=3)
    A = rng.normal(size=(3, 3))
    R = exp_so3(rng.normal(size=3))
    # Eq. (2.4): tr(x^ A) = tr(A x^) = 0.5 tr(x^(A - A^T)) = -x^T (A - A^T)^v
    lhs = np.trace(hat_so3(x) @ A)
    rhs1 = np.trace(A @ hat_so3(x))
    rhs2 = 0.5 * np.trace(hat_so3(x) @ (A - A.T))
    rhs3 = -x @ vee_so3(A - A.T)
    assert np.isclose(lhs, rhs1, atol=1e-13)
    assert np.isclose(lhs, rhs2, atol=1e-13)
    assert np.isclose(lhs, rhs3, atol=1e-13)
    # Eq. (2.5): R x^ R^T = (R x)^
    assert np.allclose(R @ hat_so3(x) @ R.T, hat_so3(R @ x), atol=1e-13)
    # Eq. (2.6): x^ A + A^T x^ = ((tr(A) I - A) x)^
    lhs_26 = hat_so3(x) @ A + A.T @ hat_so3(x)
    rhs_26 = hat_so3((np.trace(A) * np.eye(3) - A) @ x)
    assert np.allclose(lhs_26, rhs_26, atol=1e-13)
print("[PASS] Matrix identities Eq. (2.4)-(2.6)")

# Unit test 3: SO(3) exp/log round-trip (small, general, and pi)
test_phis = [
    np.zeros(3),
    np.array([1e-10, -2e-10, 3e-10]),
    np.array([0.3, -0.4, 0.5]),
    np.pi * np.array([1.0, 0.0, 0.0]),
    np.pi * np.array([1.0, -1.0, 2.0]) / np.sqrt(6.0),
]
for phi in test_phis:
    R = exp_so3(phi)
    assert np.allclose(R.T @ R, np.eye(3), atol=1e-13)
    assert np.isclose(np.linalg.det(R), 1.0, atol=1e-13)
    phi_rec = log_so3(R)
    assert np.allclose(exp_so3(phi_rec), R, atol=1e-12)
print("[PASS] SO(3) Rodrigues exp/log round-trip (including theta->0 and theta=pi)")

## 3. SE(3) Hat/Vee, Exponential/Logarithm Maps, and Adjoint

In [ ]:
def hat_se3(xi: np.ndarray) -> np.ndarray:
    """Hat map for twist xi = [v; omega] in R^6 -> se(3), Eq. (2.12)."""
    xi = np.asarray(xi, dtype=float).reshape(6)
    v, w = xi[:3], xi[3:]
    Xi = np.zeros((4, 4))
    Xi[:3, :3] = hat_so3(w)
    Xi[:3, 3] = v
    return Xi

def vee_se3(Xi: np.ndarray) -> np.ndarray:
    """Vee map se(3) -> R^6, Eq. (2.12)."""
    Xi = np.asarray(Xi, dtype=float).reshape(4, 4)
    return np.concatenate([Xi[:3, 3], vee_so3(Xi[:3, :3])])

def left_jacobian_so3(w: np.ndarray) -> np.ndarray:
    """Left Jacobian V(w) on SO(3), Eq. (2.13)."""
    theta = np.linalg.norm(w)
    K = hat_so3(w)
    if theta < 1e-8:
        t2 = theta * theta
        b = 0.5 - t2 / 24.0 + (t2 * t2) / 720.0
        c = 1.0 / 6.0 - t2 / 120.0 + (t2 * t2) / 5040.0
    else:
        b = (1.0 - np.cos(theta)) / (theta * theta)
        c = (theta - np.sin(theta)) / (theta ** 3)
    return np.eye(3) + b * K + c * (K @ K)

def left_jacobian_inv_so3(w: np.ndarray) -> np.ndarray:
    """Inverse left Jacobian V(w)^(-1) on SO(3), Eq. (2.14)."""
    theta = np.linalg.norm(w)
    K = hat_so3(w)
    if theta < 1e-8:
        t2 = theta * theta
        c = 1.0 / 12.0 + t2 / 720.0
    else:
        c = (1.0 / (theta * theta)) * (1.0 - (theta * np.sin(theta)) / (2.0 * (1.0 - np.cos(theta))))
    return np.eye(3) - 0.5 * K + c * (K @ K)

def exp_se3(xi: np.ndarray) -> np.ndarray:
    """Exponential map exp : se(3) -> SE(3), Eq. (2.13)."""
    xi = np.asarray(xi, dtype=float).reshape(6)
    v, w = xi[:3], xi[3:]
    g = np.eye(4)
    g[:3, :3] = exp_so3(w)
    g[:3, 3] = left_jacobian_so3(w) @ v
    return g

def log_se3(g: np.ndarray) -> np.ndarray:
    """Logarithm map log : SE(3) -> se(3) (returned as R^6 twist), Eq. (2.14)."""
    g = np.asarray(g, dtype=float).reshape(4, 4)
    R, p = g[:3, :3], g[:3, 3]
    w = log_so3(R)
    v = left_jacobian_inv_so3(w) @ p
    return np.concatenate([v, w])

def adjoint_se3(g: np.ndarray) -> np.ndarray:
    """Adjoint matrix Ad_g in R^(6x6) for twist [v; omega], Eq. (2.15)."""
    g = np.asarray(g, dtype=float).reshape(4, 4)
    R, p = g[:3, :3], g[:3, 3]
    Ad = np.zeros((6, 6))
    Ad[:3, :3] = R
    Ad[:3, 3:] = hat_so3(p) @ R
    Ad[3:, 3:] = R
    return Ad

# Unit test 4: SE(3) exp/log & Adjoint equivariance
for _ in range(10):
    xi = np.concatenate([rng.normal(size=3), rng.uniform(-1.5, 1.5, size=3)])
    g = exp_se3(xi)
    xi_rec = log_se3(g)
    assert np.allclose(xi_rec, xi, atol=1e-12)
    g_inv = np.linalg.inv(g)
    xi2 = rng.normal(size=6)
    assert np.allclose(vee_se3(g @ hat_se3(xi2) @ g_inv), adjoint_se3(g) @ xi2, atol=1e-12)
print("[PASS] SE(3) exp/log round-trip & Adjoint equivariance")

## 4. Quadrotor + Suspended Load Kinematics on $\mathrm{SE}(3) \times S^2$

In [ ]:
# Simulate kinematics on SE(3) x S^2: x_Q = x_L - l * q, Eq. (2.17)
dt = 0.01
T_end = 10.0
ts = np.arange(0.0, T_end + dt, dt)
l_cable = 1.0

# Load trajectory x_L(t)
x_L = np.column_stack([np.cos(ts), np.sin(ts), 0.5 * ts])

# Evolve q(t) on S^2 via q_dot = omega x q with omega . q = 0
q = np.zeros((len(ts), 3))
q[0] = np.array([0.0, 0.0, -1.0])
R_list = np.zeros((len(ts), 3, 3))
R_list[0] = np.eye(3)

q_norms = np.zeros(len(ts))
omega_dot_q = np.zeros(len(ts))
R_orth_err = np.zeros(len(ts))

for k in range(len(ts)):
    t = ts[k]
    # Tangential angular velocity on S^2: project desired rate onto T_q S^2
    w_raw = np.array([0.6 * np.cos(1.2 * t), 0.6 * np.sin(1.2 * t), 0.2])
    omega = w_raw - np.dot(w_raw, q[k]) * q[k]
    q_norms[k] = np.linalg.norm(q[k])
    omega_dot_q[k] = abs(np.dot(omega, q[k]))

    # Quadrotor body angular velocity Omega on SO(3)
    Omega = np.array([0.3 * np.sin(t), 0.3 * np.cos(t), 0.5])
    R_orth_err[k] = np.linalg.norm(R_list[k].T @ R_list[k] - np.eye(3))

    if k < len(ts) - 1:
        # Exact Lie-group integration on S^2 and SO(3)
        q[k + 1] = exp_so3(omega * dt) @ q[k]
        R_list[k + 1] = R_list[k] @ exp_so3(Omega * dt)

x_Q = x_L - l_cable * q

assert np.max(np.abs(q_norms - 1.0)) < 1e-12
assert np.max(omega_dot_q) < 1e-14
assert np.max(R_orth_err) < 1e-12
print(f"[PASS] SE(3) x S^2 kinematics invariants: max ||q||-1 = {np.max(np.abs(q_norms-1)):.2e}, max |w.q| = {np.max(omega_dot_q):.2e}")

In [ ]:
# Plot 3D trajectory and manifold invariants
fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot(x_L[:, 0], x_L[:, 1], x_L[:, 2], label='Load $x_L(t)$', linewidth=2)
ax1.plot(x_Q[:, 0], x_Q[:, 1], x_Q[:, 2], '--', label='Quadrotor $x_Q(t)$', linewidth=2)
for k in range(0, len(ts), 100):
    ax1.plot([x_Q[k, 0], x_L[k, 0]], [x_Q[k, 1], x_L[k, 1]], [x_Q[k, 2], x_L[k, 2]], 'k-', alpha=0.35)
ax1.set_title('Quadrotor + Suspended Load on $\\mathrm{SE}(3) \\times S^2$')
ax1.set_xlabel('x [m]'); ax1.set_ylabel('y [m]'); ax1.set_zlabel('z [m]')
ax1.legend()

ax2 = fig.add_subplot(1, 2, 2)
ax2.semilogy(ts, np.abs(q_norms - 1.0) + 1e-16, label='$|\\|q\\|_2 - 1|$ on $S^2$')
ax2.semilogy(ts, R_orth_err + 1e-16, label='$\\|R^T R - I_3\\|_F$ on $\\mathrm{SO}(3)$')
ax2.set_title('Manifold Invariants Under Lie-Group Integration')
ax2.set_xlabel('Time [s]'); ax2.set_ylabel('Invariant Residual')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()